In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_parquet('result/test/05_test_승인매출정보_KMeans.parquet')

df

,ID,기준년월,최종이용일자_기본,최종이용일자_CA,최종이용일자_카드론,최종이용일자_체크,최종이용일자_할부,이용건수_신용_B0M,이용건수_할부_B0M,이용건수_할부_유이자_B0M,...,이용개월수_D페이_R6M,이용금액_D페이_B0M,이용개월수_선결제_R6M,이용횟수_연체_R6M,가맹점매출금액_B1M,건수_할부전환_R6M,승인거절건수_R3M,승인거절건수_BL_R3M,승인거절건수_기타_R3M,Segment
0,TEST_00000,201807,20180731,10101,10101,10101,20160913,28,0,0,...,0,0,0,0,0,0,0,0,0,B
1,TEST_00001,201807,20180725,20170710,20171107,20180731,20180722,8,2,0,...,0,0,0,0,0,0,0,0,0,C
2,TEST_00002,201807,20180711,10101,10101,10101,20160703,59,0,0,...,0,0,0,0,6663,0,3,0,0,B
3,TEST_00003,201807,20180731,20150822,20150801,10101,20160701,37,0,0,...,0,0,0,3,0,0,0,0,0,C
4,TEST_00004,201807,20180716,20130917,10101,20180707,20180708,19,1,0,...,0,0,0,1,0,0,0,0,0,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,TEST_99995,201812,20170228,10101,10101,10101,10101,-1,0,0,...,0,0,0,0,0,0,0,0,0,A
599996,TEST_99996,201812,20181130,10101,10101,10101,10101,0,0,0,...,0,0,0,0,0,0,0,0,0,A
599997,TEST_99997,201812,10101,10101,10101,10101,10101,3,0,0,...,0,0,0,0,0,0,0,0,0,A
599998,TEST_99998,201812,20181230,10101,10101,20181225,20180608,78,0,0,...,0,0,6,0,0,0,0,0,0,E


In [6]:
required_cols = ['ID', '최종이용일자_CA', '최종이용일자_카드론, 최종이용일자_체크, 최종이용일자_할부, 이용후경과월_할부']
all(col in df.columns for col in required_cols)

False

In [9]:
missing_cols = [col for col in ['ID', '최종이용일자_CA', '최종이용일자_카드론, 최종이용일자_체크, 최종이용일자_할부, 이용후경과월_할부'] if col not in df.columns]
print("존재하지 않는 컬럼:", missing_cols)

존재하지 않는 컬럼: ['최종이용일자_카드론, 최종이용일자_체크, 최종이용일자_할부, 이용후경과월_할부']


In [3]:
# 제외할 컬럼 지정
exclude_cols = ['ID, 최종이용일자_CA']

# 수치형 컬럼에서 ID는 제외
df_numeric = df.select_dtypes(include=[np.number]).drop(columns=exclude_cols, errors='ignore').copy()

# Segment_numeric 추가
df_numeric['Segment_numeric'] = df['Segment'].astype('category').cat.codes

# Segment_numeric과의 상관계수 계산
corr_with_segment = df_numeric.corr()['Segment_numeric'].drop('Segment_numeric')

# 상관계수 ≤ 0.2인 컬럼 제거
cols_to_drop = corr_with_segment[abs(corr_with_segment) <= 0.2].index.tolist()

# 원본 df에서 제거 (ID는 유지됨)
df_filtered = df.drop(columns=cols_to_drop)

# 결과 확인
print("제거된 컬럼 수:", len(cols_to_drop))
print("남은 컬럼 수:", df_filtered.shape[1])

제거된 컬럼 수: 66
남은 컬럼 수: 6


In [4]:
df_filtered

,ID,최종이용일자_CA,최종이용일자_체크,최종이용일자_할부,이용건수_체크_B0M,Segment
0,TEST_00000,10101,10101,20160913,0,B
1,TEST_00001,20170710,20180731,20180722,22,C
2,TEST_00002,10101,10101,20160703,0,B
3,TEST_00003,20150822,10101,20160701,0,C
4,TEST_00004,20130917,20180707,20180708,11,D
...,...,...,...,...,...,...
599995,TEST_99995,10101,10101,10101,0,A
599996,TEST_99996,10101,10101,10101,0,A
599997,TEST_99997,10101,10101,10101,0,A
599998,TEST_99998,10101,20181225,20180608,55,E


In [5]:
df_filtered.to_parquet('result/test/06_test_승인매출정보_세그먼트상관계수.parquet', index=False)
df_filtered.to_csv('result/test/06_test_승인매출정보_세그먼트상관계수.csv', index=False)